# Kaze で Qwen3.8-27B を動かす

Triton 製の推論エンジン [Kaze](https://github.com/sahenjp/kaze) で、Qwen3.8-27B（hybrid 注意）を
int4 重みで読み込んで生成します。GPU は 16GB 以上を想定（T4 はぎりぎり、L4/A100 推奨）。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q torch triton transformers safetensors huggingface_hub
!pip install -q git+https://github.com/sahenjp/kaze

In [ ]:
from huggingface_hub import snapshot_download
path = snapshot_download('Qwen/Qwen3.8-27B')  # 55GB ほど落ちます
path

In [ ]:
import torch
from transformers import AutoTokenizer
from kaze.qwen38 import load_qwen38, generate

tok = AutoTokenizer.from_pretrained(path)
model = load_qwen38(path, kind='int4', dtype=torch.float16)   # int8 なら約27GB
print('GPU', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

In [ ]:
import time

prompt = '深海の調査船について、200字ほどで説明してください。'
ids = tok(prompt).input_ids
t0 = time.perf_counter()
out = generate(model, ids, max_tokens=64, temperature=0.0,
               eos_token_ids=(tok.eos_token_id,))
dt = time.perf_counter() - t0
print(tok.decode(out, skip_special_tokens=True))
print(f'--- {len(out)} tokens in {dt:.1f}s ({len(out)/dt:.1f} tok/s)')

## メモ

- decode は 1 トークンずつ Python から回しているため、まだ遅いです（CUDA Graph 未対応）。
  速さより「動くこと」を優先した実装です。
- 量子化は読み込み時にその場で行います（変換済みファイルは不要）。
- メモリが足りない場合は `kind='int8'` を `'int4'` に、`max_tokens` を小さくしてください。
